# 3 · Locked test evaluation and report (after the selection freeze)

This notebook reads the test splits. Run it for an experiment only after notebook 2 has frozen its model
selection and the freeze is recorded in `research_log/research_log.md`; every script that reads a test split
refuses an experiment without a matching `freeze.json`.

* `scripts/final_all.sh <exp>` is the whole locked evaluation of one experiment. Its views, comparison pairs
  and their order are fixed in the script (`docs/design.md` §8–9), so nothing is chosen after a test result.
  Temperatures are fitted on validation predictions, the hard pairs come from the linear head's validation
  confusions, and the "best" proposed head and control of the tables are picked on validation top-1.
* The annotation-assisted views (head_occlude, bg_gray, bg_blur, crop_body, crop_head) use the datasets' boxes.
  They are evaluated in their own pass, reported in their own columns, and never as automatic inference.
* Then: efficiency, the digest, the tables, the figures and the paper.

## Setup (run at the start of every session)

Runtime → Change runtime type → **T4 GPU** (any CUDA GPU works; the study uses a free-tier T4, 15 GB).
Mounting Google Drive is optional but strongly advised: `scripts/colab_sync.sh` mirrors everything under
`/content/work` (manifests, audits, runs, freezes, results, logs, queue state) to Drive, so a recycled runtime
loses only the feature caches. Notebook 1 rebuilds those from the images. `push()` (below) first copies every
token PCA (`pca.npz`) into `/content/work/pca_backup`, and this cell puts it back, so rebuilt caches use the
basis the heads were trained in (`extract_features.py` reuses an existing `pca.npz` instead of fitting a new one).

**Long jobs run detached.** A cell `!nohup cmd &` stays busy, and keeps the kernel busy, until the job ends,
because the job holds the cell's terminal. Every long job here is started as
`!setsid nohup bash -c '…' > /content/work/logs/<name>.log 2>&1 < /dev/null &`: the cell returns at once, and a
watch cell reads the log. The same commands can be typed into Colab's Terminal panel (without the `!`) after
exporting the variables `scripts/colab_env.py` sets, since a terminal does not have the kernel's environment.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')   # optional: skip this cell to keep everything on the runtime's disk

In [ ]:
# Where the code comes from: a git URL, or a zip of this repository (top folder DogBreed-PhD-Research)
REPO_URL = ""                                                   # e.g. the URL of your fork
REPO_ZIP = "/content/drive/MyDrive/DogBreed-PhD-Research.zip"
import os, subprocess
D = "/content/DogBreed-PhD-Research"                            # the queue files cd here: keep this path
if not os.path.isdir(D):
    if REPO_URL:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, D], check=True)
    elif os.path.isfile(REPO_ZIP):
        subprocess.run(["unzip", "-q", REPO_ZIP, "-d", "/content"], check=True)
    else:
        raise SystemExit("set REPO_URL, or put the repository zip at REPO_ZIP")
os.chdir(D)
exec(open("scripts/colab_env.py").read())                     # paths: caches on local disk, the rest on Drive
!pip install -q --no-deps -e . && python -m pytest -q tests 2>&1 | tail -2
!bash scripts/colab_sync.sh pull                                # restore earlier sessions' work, if any
!rsync -a /content/work/pca_backup/ /content/features/ 2>/dev/null && find /content/features -name pca.npz
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader


def push():
    """Copy every token PCA of the caches into /content/work/pca_backup, then mirror /content/work to Drive."""
    !rsync -am --include='*/' --include='pca.npz' --include='pca.json' --exclude='*' /content/features/ /content/work/pca_backup/
    !bash scripts/colab_sync.sh push

## Helpers

In [ ]:
import glob, json
import yaml
E, R, W = os.environ["DOGBREED_EXPERIMENTS"], os.environ["DOGBREED_RESULTS"], os.environ["WORK"]
EXPS = ["tsinghua_main", "stanford_main", "tsinghua_ablate", "tsinghua_openset", "tsinghua_resnet50",
        "tsinghua_convnext_tiny", "tsinghua_clip_vitb16", "tsinghua_tok32", "tsinghua_tok64"]


def complete(exp):
    """Every run of the experiment's config has finished for every seed: best.pt and summary.json, the two
    files scripts/freeze.py reads (train_heads.py writes best.pt just before summary.json)."""
    cfg = yaml.safe_load(open(f"configs/experiments/{exp}.yaml"))
    return all(os.path.exists(f"{E}/{exp}/{r['tag']}_s{s}/{f}") for r in cfg["runs"] for s in cfg["seeds"]
               for f in ("best.pt", "summary.json"))


def frozen(exp):
    """The experiment's freeze.json (scripts/freeze.py), or None."""
    f = f"{E}/{exp}/freeze.json"
    return json.load(open(f)) if os.path.exists(f) else None


def final_done(exp):
    """scripts/final_all.sh finished this experiment (its log ends with FINAL DONE)."""
    f = f"{W}/logs/final_{exp}.log"
    return os.path.exists(f) and f"FINAL DONE {exp}" in open(f).read()


def running(pattern):
    """A process whose command line matches (pgrep -f); write "[x]yz" so a shell never matches itself."""
    return subprocess.run(["pgrep", "-f", pattern], capture_output=True).returncode == 0

## Which experiments are frozen

In [ ]:
for exp in EXPS:
    fz = frozen(exp)
    print(f"{exp:<24}", f"frozen {fz['frozen_at_utc']} UTC, {len(fz['runs'])} runs" if fz else "NOT FROZEN",
          "| locked evaluation done" if final_done(exp) else "")

The open-set split lives in the repository checkout (`configs/openset/`), which `colab_sync.sh` does not mirror.
After a recycled runtime, write it again (same seed, same breeds) before `tsinghua_openset` is evaluated:

In [ ]:
![ -f configs/openset/tsinghua_unknown30_s0.json ] || python -I scripts/make_openset_split.py --dataset tsinghua --unknown 30 --seed 0

## The locked evaluation (detached, one experiment after another)

| Experiment | What `scripts/final_all.sh` runs, in order |
|---|---|
| `tsinghua_main` | `final_eval.py` on orig with the 22 main pairs; on the automatic corruptions (blur, noise, jpeg, lowres, occlude); separately on the annotation-assisted views (head_occlude, bg_gray, bg_blur, crop_body, crop_head), both with 7 pairs against B0 and the class query; `subset_eval.py` (the provenance subsets, orig, the 22 main pairs); `transfer_eval.py` (Tsinghua → clean Stanford test); `explain.py` (attention maps and box statistics) |
| `stanford_main` | the same three `final_eval.py` passes, with bg_gray, bg_blur, crop_body as the annotation-assisted views |
| `tsinghua_ablate` | `final_eval.py` on orig: rival vs class query at K = 2, 3, 10; pairwise vs rival at K = 3, 10 |
| `tsinghua_openset` | `ood_eval.py`: known vs held-out breeds (AUROC, FPR at 95 % TPR, closed-set top-1) |
| `tsinghua_resnet50`, `tsinghua_convnext_tiny`, `tsinghua_clip_vitb16` | `final_eval.py` on orig: rival and pairwise vs class query, pairwise vs rival, rival and class query vs B0 |
| `tsinghua_tok32`, `tsinghua_tok64` | `final_eval.py` on orig: rival vs class query, rival and class query vs the attentive probe |

Each comparison is computed per seed on the same test images (paired bootstrap, 10,000 resamples; exact
McNemar test). This cell starts every frozen experiment that has no finished evaluation yet, one after another;
each writes `/content/work/logs/final_<exp>.log`, ending with `FINAL DONE <exp>`. A failed evaluation is run
again whole the next time; record why in the research log.

In [ ]:
todo = [e for e in EXPS if frozen(e) and not final_done(e)]
chain = " ; ".join(f"bash scripts/final_all.sh {e} > /content/work/logs/final_{e}.log 2>&1" for e in todo)
if not todo:
    print("nothing to start: no frozen experiment is waiting for its locked evaluation")
elif running("[f]inal_all.sh"):
    print("a locked evaluation is still running: start this cell again when it has finished")
else:
    print("locked evaluation of:", ", ".join(todo))
    !setsid nohup bash -c '{chain}' > /content/work/logs/final_all.log 2>&1 < /dev/null &

In [ ]:
!tail -n 2 /content/work/logs/final_*.log

### Or: finish the secondary experiments unattended

The recorded run finished the ablation, open-set, backbone and token-width experiments with
`scripts/auto_final.sh` (research log, "Automatic freeze and evaluation of the secondary experiments"). Started
once beside the queues, it freezes each experiment as soon as its queue job is marked done, runs its locked
evaluation, and at the end the efficiency report, digest, tables and figures. It chooses nothing the cells here
would not; the freeze printouts it keeps in `/content/work/logs/freeze_<exp>.log` go into the research log
afterwards.

In [ ]:
if running("[a]uto_final.sh"):
    print("auto_final.sh is already running")
else:
    !setsid nohup bash -c 'bash scripts/auto_final.sh' > /content/work/logs/auto_final.log 2>&1 < /dev/null &

When every experiment's log ends with `FINAL DONE`:

In [ ]:
push()

## Efficiency (with nothing else on the GPU)

`scripts/efficiency_report.py` measures parameters, FLOPs (`torch.utils.flop_counter`; a multiply-add = 2
FLOPs), latency (median of `--iters` timed passes, default 50, after 10 warm-up passes) and peak GPU memory:
every head on random inputs of the cached shapes (batch 1 and 256, fp32), the frozen backbone (batch 1 and 64,
fp16 autocast on a T4), and the full single-image pipeline (backbone → token PCA → head). It reads no test data and
needs no freeze, but its timings only mean something when no other job uses the GPU (the first line lists the
processes on it). Output: `results/efficiency/tsinghua_main.json` and `.csv` (the paper's table reads this
experiment).

In [ ]:
!nvidia-smi --query-compute-apps=pid,process_name,used_memory --format=csv,noheader
!python -I scripts/efficiency_report.py --config configs/experiments/tsinghua_main.yaml

## Digest

`scripts/digest.py` collects every result into small CSV/JSON files in `$DOGBREED_RESULTS/digest/`, with
`MANIFEST.txt` holding the SHA-256 and size of each; the tables and figures are built from these files only.
`--curves` adds the per-epoch validation curves of the named runs. The curves figure draws seed 0 of B0, the
class query and the proposed head picked on validation, so the list holds all three proposed heads, Tsinghua
runs only.

In [ ]:
!python -I scripts/digest.py --curves tsinghua_main/linear_s0,tsinghua_main/rr_class_s0,tsinghua_main/rr_rival_s0,tsinghua_main/rr_rival_soft_s0,tsinghua_main/rr_pairwise_s0
push()

## Tables and figures (from the digest only)

`scripts/make_tables.py` writes `paper/generated/`: `tab_*.tex` (one booktabs tabular per table, input by
`paper/main.tex`), `numbers.json` (the numbers the text quotes, H1–H7 judged by the rule of `docs/design.md` §8,
and the tables skipped for missing inputs) and `figdata.json`. `scripts/make_figures.py` draws `paper/figures/`
(vector PDFs; `--png` adds previews); the attention figure also reads the maps `explain.py` saved and the
original images. Both name what they skipped.

In [ ]:
!python -I scripts/make_tables.py --digest $DOGBREED_RESULTS/digest --out paper/generated
!python -I scripts/make_figures.py --gen paper/generated --out paper/figures --explain $DOGBREED_RESULTS/tsinghua_main/explain/shown_maps.json --images /content/data/raw/tsinghua_dogs/low-resolution.zip --png

## The paper (TeX Live from apt, then latexmk)

`paper/refs.bib` is built from the two verified bibliographies by `python -I paper/merge_bib.py`. The last cell
copies the paper's inputs and the PDF into `/content/work/paper`, since `paper/` is in the repository checkout,
which `colab_sync.sh` does not mirror. The digest itself is on Drive under `work/results/digest/`: copy it into
the repository's `results/digest/` and check every file against `MANIFEST.txt` (README, "How to reproduce").

In [ ]:
!apt-get -qq update > /dev/null && apt-get -qq install -y latexmk texlive-latex-recommended texlive-latex-extra texlive-fonts-recommended > /dev/null && latexmk -v | head -n 2

In [ ]:
!cd paper && latexmk -pdf -interaction=nonstopmode -halt-on-error main.tex > /content/work/logs/latexmk.log 2>&1; tail -n 3 /content/work/logs/latexmk.log; ls -l main.pdf

In [ ]:
!mkdir -p /content/work/paper && cp -r paper/generated paper/figures paper/main.pdf /content/work/paper/
push()